In [1]:
from gitsource import GithubRepositoryDataReader
import time
reader = GithubRepositoryDataReader(

    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,

)

files = reader.read()

In [25]:
# print(files)

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [26]:
# documents

## QUESTION:1 How many lesson pages


### Q1: Answer

In [5]:
len(documents)

72

In [6]:
it = 0
for i in documents:
    it += 1
    print(f"Iteration{it}:{len(i)}")  #length of each dict
    print(f"Dict names: {i.keys()}\n") # dict names

Iteration1:2
Dict names: dict_keys(['content', 'filename'])

Iteration2:2
Dict names: dict_keys(['content', 'filename'])

Iteration3:2
Dict names: dict_keys(['content', 'filename'])

Iteration4:2
Dict names: dict_keys(['content', 'filename'])

Iteration5:2
Dict names: dict_keys(['content', 'filename'])

Iteration6:2
Dict names: dict_keys(['content', 'filename'])

Iteration7:2
Dict names: dict_keys(['content', 'filename'])

Iteration8:2
Dict names: dict_keys(['content', 'filename'])

Iteration9:2
Dict names: dict_keys(['content', 'filename'])

Iteration10:2
Dict names: dict_keys(['content', 'filename'])

Iteration11:2
Dict names: dict_keys(['content', 'filename'])

Iteration12:2
Dict names: dict_keys(['content', 'filename'])

Iteration13:2
Dict names: dict_keys(['content', 'filename'])

Iteration14:2
Dict names: dict_keys(['content', 'filename'])

Iteration15:2
Dict names: dict_keys(['content', 'filename'])

Iteration16:2
Dict names: dict_keys(['content', 'filename'])

Iteration17:2
Dic

## QUESTION: 2 Indexing and searching


In [7]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(documents)


In [8]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    boost_dict={'content':2.0, "filename":0.5},
    filter_dict={},
    num_results = 5
)

search_results

[{'content': '# The Agentic Loop\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=ePlQUcTPPjw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn the previous lesson, we did function calling by hand. We sent a\nmessage and got back a function call. We ran it, sent the result back,\nand got the answer.\n\nThat works for one function call. It breaks down when the model wants\nto search several times, or when the first search misses the answer.\nWe don\'t know in advance how many calls the model will want. So we\nneed a loop that keeps calling the model and running tools until it\'s\ndone. An agent is exactly that.\n\n## Anatomy of an agent\n\nWith the LLM in the driver\'s seat, we have an agent. It\'s an AI\nassistant whose goal is to help the user.\n\nAn agent has three parts:\n\n- Instructions, the role and behavior we want. We pass this as the\n  `developer` message. The better the instructions, the better the\n  agent helps.\n- Tools, the functions the agent can call to carry

### Q2: Answer

In [9]:
first_search_result = search_results[0]['filename']
print(first_search_result)

01-agentic-rag/lessons/14-agentic-loop.md


## QUESTION: 3 RAG

In [10]:
from openai import OpenAI
from rag_helper import RAGBase
from dotenv import load_dotenv

In [11]:
load_dotenv()
openai = OpenAI()
helper = RAGBase(
    index=index,
    llm_client=openai
)

### Q3: Answer

In [12]:
answer, token = helper.rag("How does the agentic loop keep calling the model until it stops?")
print(f"The answer: {answer}\n")
print(f"Token: {token}")

The answer: It keeps calling the model in a `while True` loop. After each response, the code checks whether the model returned any `function_call` items:

- if there are function calls, it runs the tools, appends the tool outputs to `messages`, and loops again;
- if there are no function calls, it breaks out of the loop.

So the stopping condition is: **no function calls this turn**.

Token: 7136


## QUESTION: 4 Chunking

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

### Q4: Answer

In [14]:
len(chunks)

295

## QUESTION: 5 RAG with chunking

In [15]:
from minsearch import Index

index = Index(
    text_fields=['content'],
    keyword_fields=['filename']
)
index.fit(chunks)

In [16]:
helper = RAGBase(
    index=index,
    llm_client=openai,
)

### Q5: Answer

In [17]:
chunk_answer, chunk_token = helper.rag("How does the agentic loop keep calling the model until it stops?")
print(f"Chunked Answer: {chunk_answer}\n")
print(f"Chunked Token: {chunk_token}")

Chunked Answer: The loop keeps calling the model in a `while True` loop. After each model response, it checks whether there were any `function_call` items.

- If there was at least one function call, it runs the tool, appends the result to `messages`, and loops again.
- If there were no function calls, it breaks out of the loop.

So the stop condition is:

```python
if has_function_calls == False:
    break
```

In other words, it keeps going until the model returns a final answer with no more tool calls.

Chunked Token: 2319


## QUESTION: 6 Turning it into an agent

In [18]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [19]:
instructions = "You're a course teaching assistant. Answer the student's question using the search tool. Make multiple searches with different keywords before answering."

In [20]:
def search(query: str) -> dict[str, str]:
    """
    Search the course lessons database for entries matching the given query.
    """
    return index.search(
        query,
        num_results=5,
        boost_dict={"question":3.0, "section": 0.5},
        filter_dict={"course": "llm-zoomcamp"}
    )



In [21]:
agent_tools = Tools()
agent_tools.add_tool(search)


In [22]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'query parameter'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [23]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

runner = OpenAIResponsesRunner(
    tools = agent_tools,
    developer_prompt = instructions,
    chat_interface=chat_interface,
    llm_client = OpenAIClient(model="gpt-5.4-mini")
)


### Q6: Answer

In [24]:
result = runner.loop(
    prompt="How does the agentic loop work, and how is it different from plain RAG?",
    callback=callback,
)

-> Response received


-> Response received


-> Response received
